#### Feature Engineering

Dataset:IEEE-CIS Fraud Detection

| Step | Note |
|------|------|
| Memory Optimization | Downcasts data types to reduce memory usage |
| Data Loading | Merges transaction and identity data, sorts by time |
| Baseline Cleaning | Maps email domains, adds time features (hour/day), label encodes categorical variables |
| Domain-Specific Features | Adds aggregated statistics (mean/std transaction amounts per card), spending deviations, transaction velocity, and device-card aggregations |
| Output | Saves baseline and engineered datasets to Parquet files |

In [ ]:
import pandas as pd
import numpy as np
import gc
import warnings
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

DATA_DIR = 'data_raw/IEEE-CIS/ieee-fraud-detection'

# 1. MEMORY OPTIMIZATION
def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Mem decreased to {end_mem:.2f} Mb ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    return df

# 2. LOAD DATA
print("Loading data...")
train_trans = pd.read_csv(f'{DATA_DIR}/train_transaction.csv')
train_id = pd.read_csv(f'{DATA_DIR}/train_identity.csv')
df = pd.merge(train_trans, train_id, on='TransactionID', how='left')

# Sort by Time (Chronological Splitting Requirement)
df = df.sort_values('TransactionDT').reset_index(drop=True)

# 3. BASELINE CLEANING 
print("Applying baseline cleaning...")
# Email domain mapping
emails = {'gmail': 'google', 'att.net': 'at&t', 'twc.com': 'spectrum', 'sc.rr.com': 'spectrum', 
          'nycap.rr.com': 'spectrum', 'charter.net': 'spectrum', 'prodigy.net.mx': 'at&t', 
          'protonmail.com': 'protonmail', 'ptd.net': 'penn_telecom', 'aim.com': 'aol', 
          'web.de': 'other', 'icloud.com': 'apple', 'hotmail.com': 'microsoft'}

for c in ['P_emaildomain', 'R_emaildomain']:
    df[c] = df[c].map(emails).fillna('other')

# Time Features
df['Transaction_hour'] = np.floor(df['TransactionDT'] / 3600) % 24
df['Transaction_day'] = np.floor(df['TransactionDT'] / (3600 * 24)) % 7

# Label Encoding
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# Create baseline parquet before adding domain features
df_baseline = reduce_mem_usage(df.copy())
df_baseline.to_parquet('data/iceee_baseline.parquet')

# 4. DOMAIN-SPECIFIC FEATURE ENGINEERING (Rubric Requirements)
print("Adding Domain-Specific Features...")

# A. AGGREGATED STATISTICS (User-level behavior)
# Grouping by card1 (often used as a proxy for user)
df['card1_TransactionAmt_mean'] = df.groupby(['card1'])['TransactionAmt'].transform('mean')
df['card1_TransactionAmt_std'] = df.groupby(['card1'])['TransactionAmt'].transform('std')

# B. SPENDING PATTERN DEVIATIONS
# Ratio of current transaction to user average
df['Amt_to_mean_card1'] = df['TransactionAmt'] / df['card1_TransactionAmt_mean']

# C. TRANSACTION VELOCITY
# Count transactions per card1 in a specific timeframe (approximated by index window for simplicity)
df['card1_count'] = df.groupby(['card1'])['TransactionID'].transform('count')

# D. NEW AGGREGATIONS (Cross-referencing Device and Card)
# Number of unique devices used per card
df['card1_Device_nuniq'] = df.groupby(['card1'])['DeviceInfo'].transform('nunique')

# Fill new NaNs and optimize
df = df.fillna(-999)
df_engineered = reduce_mem_usage(df)

# 5. SAVE FINAL DATASETS
print("Saving engineered parquet...")
df_engineered.to_parquet('data/iceee_feature.parquet')

print("Process Complete. Produced: iceee_baseline.parquet and iceee_feature.parquet")

Loading data...
Applying baseline cleaning...
Mem decreased to 928.69 Mb (52.7% reduction)
Adding Domain-Specific Features...
Mem decreased to 937.70 Mb (52.8% reduction)
Saving engineered parquet...
Process Complete. Produced: iceee_baseline.parquet and iceee_feature.parquet
